<a href="https://colab.research.google.com/github/menna-em3/-/blob/main/virtual_robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from enum import Enum
import time

## Environment

In [14]:
class CellType(Enum):
    EMPTY = 0
    OBSTACLE = 1
    GOAL = 2

class Environment:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = [[CellType.EMPTY for _ in range(width)]
                     for _ in range(height)]

    def is_within_bounds(self, x, y):
        return 0 <= x < self.width and 0 <= y < self.height

    def is_obstacle(self, x, y):
        return self.grid[y][x] == CellType.OBSTACLE

    def is_goal(self, x, y):
        return self.grid[y][x] == CellType.GOAL

    def display(self, robot=None):
        for y in range(self.height):
            for x in range(self.width):
                if robot and robot.x == x and robot.y == y:
                    print("R", end=" ")
                elif self.grid[y][x] == CellType.OBSTACLE:
                    print("#", end=" ")
                elif self.grid[y][x] == CellType.GOAL:
                    print("G", end=" ")
                else:
                    print(".", end=" ")
            print()
        print()



## Robot

In [15]:
import random

In [16]:
class Direction(Enum):
    NORTH = 0
    EAST = 1
    SOUTH = 2
    WEST = 3

class RobotState(Enum):
    IDLE = 0
    MOVING = 1
    AVOIDING = 2
    FINISHED = 3

class Robot:
    def __init__(self, x, y, direction=Direction.NORTH):
        self.x = x
        self.y = y
        self.direction = direction
        self.state = RobotState.IDLE

    def turn_left(self):
        self.direction = Direction((self.direction.value - 1) % 4)

    def turn_right(self):
        self.direction = Direction((self.direction.value + 1) % 4)

    def next_position(self):
        dx, dy = 0, 0
        if self.direction == Direction.NORTH:
            dy = -1
        elif self.direction == Direction.SOUTH:
            dy = 1
        elif self.direction == Direction.EAST:
            dx = 1
        elif self.direction == Direction.WEST:
            dx = -1
        return self.x + dx, self.y + dy

    def move_forward(self, environment):
        nx, ny = self.next_position()

        if not environment.is_within_bounds(nx, ny):
            self.state = RobotState.AVOIDING
            return False

        if environment.is_obstacle(nx, ny):
            self.state = RobotState.AVOIDING
            return False

        self.x = nx
        self.y = ny
        self.state = RobotState.MOVING
        return True

    def sense(self, environment):
        nx, ny = self.next_position()
        if not environment.is_within_bounds(nx, ny):
            return "WALL"
        if environment.is_obstacle(nx, ny):
            return "OBSTACLE"
        if environment.is_goal(nx, ny):
            return "GOAL"
        return "CLEAR"

    def decide(self, sensor_data, environment):
        """FSM-based decision making"""
        if sensor_data == "GOAL":
            self.state = RobotState.FINISHED
            print("Goal reached!")
            return

        if self.state in [RobotState.IDLE, RobotState.MOVING]:
            if sensor_data == "CLEAR":
                self.move_forward(environment)
            elif sensor_data in ["OBSTACLE", "WALL"]:
                self.state = RobotState.AVOIDING

        elif self.state == RobotState.AVOIDING:

            turn_choice = random.choice(["LEFT", "RIGHT"])
            if turn_choice == "LEFT":
                self.turn_left()
            else:
                self.turn_right()

            if self.sense(environment) == "CLEAR":
                self.move_forward(environment)
                self.state = RobotState.MOVING
            else:

                self.state = RobotState.AVOIDING


## Simulation

In [17]:
class Simulation:
    def __init__(self, environment, robot, max_steps=100):
        self.environment = environment
        self.robot = robot
        self.max_steps = max_steps
        self.current_step = 0

    def step(self):
        sensor_data = self.robot.sense(self.environment)
        self.robot.decide(sensor_data, self.environment)

        if self.robot.state == RobotState.FINISHED:
            return False

        self.current_step += 1
        return self.current_step < self.max_steps

    def run(self, delay=0.3):
        running = True
        while running:
            self.environment.display(self.robot)
            running = self.step()
            time.sleep(delay)

        print("Simulation finished.")


## Main

In [18]:
def main():
    env = Environment(width=10, height=10)


    env.grid[3][3] = CellType.OBSTACLE
    env.grid[4][3] = CellType.OBSTACLE
    env.grid[7][7] = CellType.GOAL

    robot = Robot(x=0, y=0)

    sim = Simulation(env, robot)
    sim.run()


if __name__ == "__main__":
    main()

R . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . # . . . . . . 
. . . # . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . G . . 
. . . . . . . . . . 
. . . . . . . . . . 

R . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . # . . . . . . 
. . . # . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . G . . 
. . . . . . . . . . 
. . . . . . . . . . 

. R . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . # . . . . . . 
. . . # . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . G . . 
. . . . . . . . . . 
. . . . . . . . . . 

. . R . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . # . . . . . . 
. . . # . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . G . . 
. . . . . . . . . . 
. . . . . . . . . . 

. . . R . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . # . . . . . . 
. . . # . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . .